# 需求描述=
1获取视频的宽高，提取宽高为（竖屏）视频到新建的文件夹，保持文件相对路径。
2另外一个文件夹里是视频文件，提取其中的音频，将声音提高15分贝，
3计算文件夹内视频文件总时长，多少种宽高，将不同分辨率的文件移动进以分辨率命名的文件夹里
4取文件夹A内文件的音频，将声音提高15分贝。跟文件夹B内文件的视频不要音频，二者拼接，生成文件放入文件夹C，以文件夹A内文件名来命名。print打印程序运行流程进度。实现逻辑，以A内文件为主，每分离一个A内的文件，从B内截取出相应长度的无音频视频。实现一个B里视频的缓存池队列，A需要视频时，从这个队列里取出一定长度的视频，并补充一定长度的B里的视频，在需要拿一个新的B文件时，随机取文件，B里视频只用一次，定时打印进度。
5注意最后的边界，已知A音频总长度长于B视频总长度。最后必然是B视频用完了，这时A音频必然有部分无对应的视频，处理方法是，再从B中随机取出A缺少的长度的视频补上
6合并输出文件太大了，增加以下要求：使用 H.264 编码器、设置视频帧率30、设置视频分辨率1920x1080、设置视频比特率（VBR）3Mbps、使用 AAC 音频编码、设置音频比特率128kbps、设置为立体声（2声道）、设置音频采样率为 44100Hz、允许使用实验性功能。一个A内文件对应一个生成文件，当前一个90分钟的视频大小超过5G，这很不合理。
7当前是单线程任务，在缓存文件中生成的文件还要复制为目标文件，这些复制操作受限于硬盘读写速度。现在修改为将C盘D文件夹作为工作目录，将缓存文件等放进工作目录，当文件生成完成，需要将缓存复制成目标文件时，开启另一个线程来完成复制任务，并且将文件复制到C文件夹中。此外，缓存文件中的mp4是否就等同于生成的目标文件，是否只需要改个名字就行。
8不要D文件夹了，缓存文件还是放在C文件里吧，我的资源文件都在G盘，要将资源复制到C盘，这速度更低
9日志打印太少了，在执行cmd程序前后也加上日志，还有其他有必要的地方都加上，用print打日志，将不变的量作为写进缓存txt文件里，我要看
10设置输出视频的那些参数需要ffmpeg进行大量计算，舍弃这个要求，将视频不做编码处理，保持原样，只做截取和拼接处理。注意一个B文件的视频，可能与多个A文件音频生成多个C文件目标视频
------11处理A文件音频的同时合并B文件视频。不生成缓存文件
12生成的目标文件确实合并成功了，但是你没处理缓存文件，请加上处理缓存文件的逻辑，比如在处理完某个目标文件后删除上上个目标文件产生的缓存文件
13第二个目标文件的视频内容跟第一个目标文件的一致，原因是，虽然视频长度有音频长度决定的，但是视频开始的起点是一致的，要解决这个问题，在第一个目标文件生成后，记录B文件中视频文件的时间点，往前提前1~3秒，作为第二个目标文件的视频文件的起点。注意边界，如果视频前面没有可提前的余量也可以不提前
14生成文件有问题，当A音频文件对应的视频文件不够时，需要B的队列里新增文件，跟队列上一个文件的剩余部分进行拼接。当前这种情况出现时，视频是损坏的。最后，输出每个音频文件对应的视频文件时间点范围，写进缓存txt文件里我要看，这种txt文件可以作为不变量的记录，我要看
15视频还是没有拼接上，生成文件的后面那段应该拼接下一个B文件夹视频才对。现在换一个策略，维护两条时间轴，A文件音频轴，B文件视频轴。记录时间节点，输出B文件视频需要截断的时间点。先计算这些，不要一开始就开始生成文件。先将结果写入到缓存txt文件中，你之前的txt文件都是空的，没有写入东西。结果格式为，A文件音频名称-长度，需要B文件视频名称，起始点到截断点；A文件音频名称2-长度，需要B文件视频名称，起始点到截断点；输出格式美观点。输出完之后，再根据生成的txt文件开始拼接。

MP3
文件夹内视频总时长: 113.39 小时 (408204.27 秒)
共有 1 种不同的宽高分辨率: {(256, 144)}

MP4
文件夹内视频总时长: 94.60 小时 (340571.58 秒)
共有 3 种不同的宽高分辨率: {(1920, 1080), (1280, 720), (7680, 4320)}

In [ ]:
# 2另外一个文件夹里是视频文件，提取其中的音频，将声音提高15分贝，
import os
import subprocess

# 输入和输出文件夹（请替换为你的实际路径）
input_folder = ""  # MP4 视频所在的文件夹
output_folder = ""  # 输出音频的文件夹

# 确保输出文件夹存在
os.makedirs(output_folder, exist_ok=True)

# 遍历输入文件夹中的所有 MP4 文件
for filename in os.listdir(input_folder):
    if filename.lower().endswith(".mp4"):  # 忽略大小写
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, os.path.splitext(filename)[0] + ".mp3")  # 生成 MP3 文件

        # 确保源文件不会被删除，FFmpeg 只提取音频
        command = [
            "ffmpeg", "-i", input_path, "-vn", "-af", "volume=15dB",
            "-y", output_path
        ]

        # 执行 FFmpeg 命令
        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("音频提取并增益完成！")


In [ ]:
# 3计算文件夹内视频文件总时长，多少种宽高
import os
import subprocess

# 输入文件夹路径（请替换为你的实际路径）
input_folder = ""

# 存储总时长（秒）和不同的分辨率集合
total_duration = 0
resolutions = set()

# 遍历文件夹中的所有视频文件
for filename in os.listdir(input_folder):
    if filename.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".flv")):  # 过滤常见视频格式
        input_path = os.path.join(input_folder, filename)

        # 获取视频时长（秒）
        cmd_duration = [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "format=duration", "-of", "csv=p=0", input_path
        ]
        result = subprocess.run(cmd_duration, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        try:
            duration = float(result.stdout.strip())  # 解析时长
            total_duration += duration
        except ValueError:
            print(f"无法获取 {filename} 的时长")

        # 获取视频分辨率（宽×高）
        cmd_resolution = [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=width,height", "-of", "csv=p=0", input_path
        ]
        result = subprocess.run(cmd_resolution, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        try:
            width, height = map(int, result.stdout.strip().split(","))
            resolutions.add((width, height))  # 添加到分辨率集合
        except ValueError:
            print(f"无法获取 {filename} 的分辨率")

# 输出统计结果
print(f"文件夹内视频总时长: {total_duration / 3600:.2f} 小时 ({total_duration:.2f} 秒)")
print(f"共有 {len(resolutions)} 种不同的宽高分辨率: {resolutions}")


In [ ]:
# 3将不同分辨率的文件移动进以分辨率命名的文件夹里
import os
import shutil
import subprocess

# 输入和输出文件夹（请修改为实际路径）
input_folder = "G:\XOXO主播\long"
output_folder = "G:\XOXO主播\long"

# 确保输出文件夹存在
os.makedirs(output_folder, exist_ok=True)

def is_file_in_use(file_path):
    """检查文件是否正在被使用"""
    try:
        with open(file_path, "rb") as f:
            pass  # 能正常打开文件，说明没有被占用
        return False
    except IOError:
        return True  # 文件正在被占用

# 遍历所有视频文件
for filename in os.listdir(input_folder):
    if filename.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".flv")):  # 过滤常见视频格式
        input_path = os.path.join(input_folder, filename)

        # 检查文件是否正在被使用
        if is_file_in_use(input_path):
            print(f"跳过（文件正在使用）: {filename}")
            continue

        # 获取视频分辨率，限制 5 秒超时
        cmd_resolution = [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=width,height", "-of", "csv=p=0", input_path
        ]
        try:
            result = subprocess.run(cmd_resolution, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5)
            width, height = map(int, result.stdout.strip().split(","))

            # 创建分辨率文件夹
            resolution_folder = os.path.join(output_folder, f"{width}x{height}")
            os.makedirs(resolution_folder, exist_ok=True)

            # 移动文件
            shutil.move(input_path, os.path.join(resolution_folder, filename))
            print(f"已移动: {filename} → {resolution_folder}/")

        except subprocess.TimeoutExpired:
            print(f"跳过（FFprobe 超时）: {filename}")

        except ValueError:
            print(f"跳过（无法获取分辨率）: {filename}")

print("文件整理完成！")


# Gen

In [ ]:
# 3-14--11
import os
import random
import subprocess
import time

# 文件夹路径
# folder_A = "G:/path/to/folder_A"  # 音频来源
# folder_B = "G:/path/to/folder_B"  # 视频来源
# folder_C = "C:/path/to/folder_C"  # 目标文件夹
cache_txt = os.path.join(folder_C, "cache_ranges.txt")  # 记录时间范围

# 确保目标文件夹存在
os.makedirs(folder_C, exist_ok=True)

# 获取文件列表
files_A = sorted(f for f in os.listdir(folder_A) if f.lower().endswith(".mp4"))
files_B = [f for f in os.listdir(folder_B) if f.lower().endswith(".mp4")]
random.shuffle(files_B)  # 随机打乱 B 文件顺序

b_video_queue = []  # B 视频缓存队列
used_b_files = {}  # 记录 B 文件的使用时间点
cache_files = []  # 记录缓存文件，便于删除
time_ranges = []  # 记录A文件的B视频时间范围


def get_video_duration(video_path):
    """获取视频时长"""
    cmd = ["ffprobe", "-v", "error", "-select_streams", "v:0",
           "-show_entries", "format=duration", "-of", "csv=p=0", video_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5)
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None


def extract_audio(input_video, output_audio):
    """提取音频并增加 15dB"""
    cmd = ["ffmpeg", "-i", input_video, "-vn", "-filter:a", "volume=15dB",
           "-c:a", "aac", "-b:a", "128k", "-ac", "2", "-ar", "44100", "-y", output_audio]
    print(f"🔊 提取音频: {cmd}")
    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)


def cut_video(input_video, output_video, start_time, duration):
    """截取视频"""
    cmd = ["ffmpeg", "-i", input_video, "-ss", str(start_time), "-t", str(duration),
           "-c:v", "copy", "-an", "-y", output_video]
    print(f"✂️ 截取视频: {cmd}")
    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)


def merge_video_audio(input_video_list, input_audio, output_video):
    """合并多个视频和音频"""
    input_videos = "concat:" + "|".join(input_video_list)
    cmd = ["ffmpeg", "-i", input_videos, "-i", input_audio,
           "-map", "0:v:0", "-map", "1:a:0",
           "-c:v", "copy", "-c:a", "aac", "-b:a", "128k", "-ac", "2", "-ar", "44100",
           "-y", output_video]
    print(f"🎬 合并音视频: {cmd}")
    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)


def delete_old_cache():
    """删除上上个目标文件的缓存"""
    if len(cache_files) > 2:
        old_cache = cache_files.pop(0)
        for file in old_cache:
            if os.path.exists(file):
                os.remove(file)
                print(f"🗑️ 删除缓存: {file}")


with open(cache_txt, "w") as f_cache:
    for index, file_A in enumerate(files_A, start=1):
        input_audio_path = os.path.join(folder_A, file_A)
        output_video_path = os.path.join(folder_C, file_A)  # 目标文件
        temp_audio_path = os.path.join(folder_C, f"{file_A}_audio.aac")
        temp_video_parts = []  # 存放多个视频片段路径
        video_time_range = []  # 记录当前 A 文件用到的 B 视频时间范围

        print(f"\n[{index}/{len(files_A)}] 处理 {file_A}...")

        # 删除上上个文件的缓存
        delete_old_cache()

        # 1️⃣ 获取 A 音频时长
        duration_A = get_video_duration(input_audio_path)
        if duration_A is None:
            print(f"❌ 跳过（无法获取时长）: {file_A}")
            continue

        # 2️⃣ 提取 A 音频
        extract_audio(input_audio_path, temp_audio_path)

        # 3️⃣ 获取视频片段
        collected_duration = 0

        while collected_duration < duration_A:
            if not b_video_queue:  # 队列为空时补充
                while files_B:
                    b_file = files_B.pop()
                    if b_file in used_b_files:
                        continue

                    b_path = os.path.join(folder_B, b_file)
                    duration_B = get_video_duration(b_path)
                    if duration_B is None:
                        continue

                    used_b_files[b_path] = 0  # 初始化使用时间点
                    b_video_queue.append({"path": b_path, "duration": duration_B, "start": 0})
                    break

            if not b_video_queue:  # B 资源用尽
                print(f"⚠️ B 文件用尽，无法继续合成 {file_A}")
                break

            # 取出 B 视频
            current_video = b_video_queue[0]
            b_path = current_video["path"]
            start_time = used_b_files.get(b_path, 0)

            # 计算提前量
            if start_time > 3:
                start_time -= random.randint(1, 3)

            # 计算剩余时长
            remaining_time = current_video["duration"] - start_time
            clip_duration = min(remaining_time, duration_A - collected_duration)

            temp_video_path = os.path.join(folder_C, f"{file_A}_part{len(temp_video_parts)}.mp4")
            cut_video(b_path, temp_video_path, start_time, clip_duration)
            temp_video_parts.append(temp_video_path)
            video_time_range.append(f"{b_path} [{start_time} - {start_time + clip_duration}]")

            collected_duration += clip_duration
            used_b_files[b_path] = start_time + clip_duration

            if used_b_files[b_path] >= current_video["duration"]:
                b_video_queue.pop(0)  # 该文件用完，移出队列

        # 4️⃣ 合并视频
        merge_video_audio(temp_video_parts, temp_audio_path, output_video_path)

        # 记录缓存
        cache_files.append([temp_audio_path] + temp_video_parts)

        # 记录时间范围
        f_cache.write(f"{file_A}:\n" + "\n".join(video_time_range) + "\n\n")

        print(f"✅ 处理完成: {output_video_path}\n")

# 最后删除多余缓存
while cache_files:
    delete_old_cache()

print("🎉 任务完成！")


# Gen2

In [ ]:
# 11处理A文件音频的同时合并B文件视频。不生成缓存文件
import os
import random
import subprocess

# 文件夹路径
# folder_A = "G:/path/to/folder_A"  # 音频来源
# folder_B = "G:/path/to/folder_B"  # 视频来源
# folder_C = "C:/path/to/folder_C"  # 目标文件夹

# 确保目标文件夹存在
os.makedirs(folder_C, exist_ok=True)

# 获取 A 和 B 的文件列表
files_A = sorted(f for f in os.listdir(folder_A) if f.lower().endswith(".mp4"))
files_B = [f for f in os.listdir(folder_B) if f.lower().endswith(".mp4")]
random.shuffle(files_B)  # 随机打乱 B 文件顺序

b_video_queue = []  # B 视频缓存队列
used_b_files = set()  # 已使用的 B 文件


def get_video_duration(video_path):
    """获取视频时长"""
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "format=duration", "-of", "csv=p=0", video_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None


for index, file_A in enumerate(files_A, start=1):
    input_audio_path = os.path.join(folder_A, file_A)
    output_path = os.path.join(folder_C, file_A)

    print(f"[{index}/{len(files_A)}] 处理 {file_A}...")

    # 1️⃣ 获取 A 文件音频的时长
    duration_A = get_video_duration(input_audio_path)
    if duration_A is None:
        print(f"❌ 跳过（无法获取时长）: {file_A}")
        continue

    # 2️⃣ 获取所需的视频片段
    collected_duration = 0
    video_segments = []
    
    while collected_duration < duration_A:
        if not b_video_queue:  # 队列为空时补充
            while files_B:
                b_file = files_B.pop()
                if b_file in used_b_files:
                    continue
                
                b_path = os.path.join(folder_B, b_file)
                duration_B = get_video_duration(b_path)
                if duration_B is None:
                    continue

                used_b_files.add(b_file)
                b_video_queue.append({"path": b_path, "duration": duration_B})
                break  # 只补充一个新视频

        if not b_video_queue:  # B 资源全部用完
            print(f"⚠️ B 文件用尽，无法继续合成 {file_A}")
            break

        # 取出 B 队列的视频
        current_video = b_video_queue[0]
        video_segments.append(f"file '{current_video['path']}'")
        collected_duration += current_video["duration"]

        # 如果 B 视频完全使用过了，则移除
        if collected_duration >= current_video["duration"]:
            b_video_queue.pop(0)

    if collected_duration < duration_A:
        print(f"⚠️ 无足够 B 视频，跳过 {file_A}")
        continue

    # 3️⃣ 构建 ffmpeg 直接管道
    print(f"🎬 直接处理 {file_A}，不生成中间文件...")

    # 构造 concat 命令
    concat_cmd = " | ".join([f"echo {seg}" for seg in video_segments])

    ffmpeg_cmd = f"""
        ( {concat_cmd} ) |
        ffmpeg -f concat -safe 0 -i - -i "{input_audio_path}" 
        -map 0:v:0 -map 1:a:0 
        -filter:a "volume=15dB" 
        -c:v copy -c:a aac -b:a 128k -ac 2 -ar 44100 
        -strict experimental -y "{output_path}"
    """

    print(f"执行命令: {ffmpeg_cmd}")
    
    try:
        subprocess.run(ffmpeg_cmd, shell=True, check=True)
        print(f"✅ 处理完成: {output_path}")
    except subprocess.CalledProcessError:
        print(f"❌ 失败: {file_A}")

    print(f"进度：{index}/{len(files_A)} 处理完毕")

print("🎉 任务完成！")


# Gen3

In [ ]:
# 14采用“先计算资源分配计划，再进行实际处理”的策略：
import os
import random
import subprocess

from datetime import timedelta

# 文件夹路径
# folder_A = "G:/path/to/folder_A"  # 音频来源
# folder_B = "G:/path/to/folder_B"  # 视频来源
# folder_C = "C:/path/to/folder_C"  # 输出记录文件

os.makedirs(folder_C, exist_ok=True)

plan_txt = os.path.join(folder_C, "merge_plan.txt")

def get_video_duration(path):
    try:
        cmd = [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "format=duration", "-of", "csv=p=0", path
        ]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5)
        return float(result.stdout.strip())
    except Exception:
        return None

def format_time(seconds):
    return str(timedelta(seconds=round(seconds, 2)))

# 收集文件
files_A = sorted(f for f in os.listdir(folder_A) if f.lower().endswith(".mp4"))
files_B = sorted(f for f in os.listdir(folder_B) if f.lower().endswith(".mp4"))
random.shuffle(files_B)

# 时间轴初始化
b_timeline = []  # list of dicts: {"path": str, "duration": float, "used": float}
for b_file in files_B:
    b_path = os.path.join(folder_B, b_file)
    dur = get_video_duration(b_path)
    if dur:
        b_timeline.append({"file": b_file, "duration": dur, "used": 0})

# 创建计划文件
with open(plan_txt, "w", encoding="utf-8") as f:
    for a_file in files_A:
        a_path = os.path.join(folder_A, a_file)
        a_dur = get_video_duration(a_path)
        if not a_dur:
            print(f"❌ 跳过无效文件: {a_file}")
            continue

        f.write(f"{a_file}（音频长度：{format_time(a_dur)}）\n")

        remaining = a_dur
        plan = []

        while remaining > 0 and b_timeline:
            current = b_timeline[0]
            b_file = current["file"]
            available = current["duration"] - current["used"]

            if available <= 0:
                b_timeline.pop(0)
                continue

            use_time = min(remaining, available)
            start = current["used"]
            end = start + use_time
            plan.append((b_file, start, end))

            current["used"] += use_time
            remaining -= use_time

        # 输出分配计划
        if not plan:
            f.write("⚠️ 无足够视频资源可用！\n")
        else:
            for b_file, start, end in plan:
                f.write(f"  使用文件 {b_file}：{format_time(start)} - {format_time(end)}\n")
        f.write("\n")

print(f"✅ 资源分配计划已生成：{plan_txt}")


✅ 资源分配计划已生成：G:\05ClipMP4\003mpHongLou\merge_plan.txt


# 方案变更=
15moviepy.editor这个是干什么用的之前你没用过这个库。如果不是错误的话可以保留，
16可以尝试优化代码提升性能，对于新生成的C文件夹文件，可以保留上一个生成文件末尾的视频1~3秒，可以在视频上衔接上，对于音频不做这种要求。
17要做到输出文件是A音频跟B视频合并成的，不要让输出文件只有音频或者只有视频。
18针对缓存文件，可以做缓存，但是要及时清理避免硬盘空间不足
19注意要有必要的日志打印
20AB文件夹里的文件都是mp4，只是需要提取其中的音频和视频。然后在生成目标文件之前，采用“先计算资源分配计划，再进行实际处理”的策略，输出一个txt文件记录
21注意边界条件，首次拿到的duration_str是空的，输出的目标文件名跟A文件名保持一致

明天需补充
22注意B的每个文件只能用一次，当B文件都用完时，且A里还有个音频文件只有部分视频，可以在B文件里随机截取相应的视频补给A的那个音频文件，再后面，A里还没有匹配视频的音频文件，输出文件名告知用户即可。

In [ ]:
# 15moviepy编译报错
import os
import random
from moviepy.video.io.VideoFileClip import VideoFileClip
from moviepy.audio.io.AudioFileClip import AudioFileClip
from moviepy.video.compositing.CompositeVideoClip import concatenate_videoclips

# 假设A音频文件夹路径、B视频文件夹路径和C输出文件夹路径
# A_folder = "path/to/A"
# B_folder = "path/to/B"
# C_folder = "path/to/C"

# 获取A和B文件夹中的文件
def get_files_in_folder(folder):
    return [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.mp4')]

# 获取视频文件的时长（秒）
def get_video_duration(video_path):
    with VideoFileClip(video_path) as video:
        return video.duration

# 获取音频文件的时长（秒）
def get_audio_duration(audio_path):
    with AudioFileClip(audio_path) as audio:
        return audio.duration

# 生成目标视频文件的输出路径
def get_output_filename(audio_filename):
    return os.path.join(C_folder, f"{os.path.splitext(audio_filename)[0]}_output.mp4")

# 合并音频和视频
def merge_audio_video(audio_path, video_path, start_time, duration, output_filename):
    with AudioFileClip(audio_path) as audio:
        with VideoFileClip(video_path) as video:
            video = video.subclipped(start_time, start_time + duration)  # 截取视频
            video = video.set_audio(audio.subclipped(0, video.duration))  # 设置音频
            video.write_videofile(output_filename, codec='libx264', audio_codec='aac', threads=4)

# 计算每个音频文件需要的视频片段
def calculate_timeline(A_files, B_files):
    timeline_info = []
    current_b_file_index = 0
    current_b_start_time = 0

    for audio_path in A_files:
        audio_duration = get_audio_duration(audio_path)
        audio_filename = os.path.basename(audio_path)

        remaining_audio_duration = audio_duration
        current_b_file_index = random.choice(range(len(B_files)))  # 随机选择B文件

        while remaining_audio_duration > 0:
            video_path = B_files[current_b_file_index]
            video_duration = get_video_duration(video_path)

            if remaining_audio_duration <= video_duration:
                timeline_info.append({
                    "audio_filename": audio_filename,
                    "audio_duration": audio_duration,
                    "video_filename": os.path.basename(video_path),
                    "start_time": current_b_start_time,
                    "end_time": current_b_start_time + remaining_audio_duration
                })

                remaining_audio_duration = 0
            else:
                timeline_info.append({
                    "audio_filename": audio_filename,
                    "audio_duration": audio_duration,
                    "video_filename": os.path.basename(video_path),
                    "start_time": current_b_start_time,
                    "end_time": current_b_start_time + video_duration
                })

                remaining_audio_duration -= video_duration
                current_b_start_time += video_duration
                current_b_file_index = (current_b_file_index + 1) % len(B_files)  # 切换到下一个B文件

    return timeline_info

# 将时间轴信息写入缓存文件
def write_to_cache_file(timeline_info, cache_file_path):
    with open(cache_file_path, 'w') as cache_file:
        for entry in timeline_info:
            cache_file.write(f"A音频: {entry['audio_filename']} - 长度: {entry['audio_duration']}秒\n")
            cache_file.write(f"B视频: {entry['video_filename']} - 起始点: {entry['start_time']}秒, 截断点: {entry['end_time']}秒\n")
            cache_file.write("\n")

# 处理所有A音频文件与B视频文件进行合并
def process_all_files(A_files, B_files, timeline_info):
    for entry in timeline_info:
        audio_filename = entry['audio_filename']
        video_filename = entry['video_filename']
        start_time = entry['start_time']
        end_time = entry['end_time']

        output_filename = get_output_filename(audio_filename)
        merge_audio_video(os.path.join(A_folder, audio_filename),
                          os.path.join(B_folder, video_filename),
                          start_time, end_time - start_time,
                          output_filename)
        print(f"已生成：{output_filename}")

if __name__ == "__main__":
    # 获取A和B文件夹内的文件
    A_files = get_files_in_folder(A_folder)
    B_files = get_files_in_folder(B_folder)

    # 计算时间轴
    timeline_info = calculate_timeline(A_files, B_files)

    # 输出缓存文件
    cache_file_path = "cache_info.txt"
    write_to_cache_file(timeline_info, cache_file_path)
    print(f"时间轴信息已写入 {cache_file_path}")

    # 根据缓存文件生成目标视频文件
    process_all_files(A_files, B_files, timeline_info)


# Gen-final

In [ ]:
#1-21所有任务已完成
import os
import subprocess
import random
import threading
from pathlib import Path
from typing import List, Tuple

# A_DIR = Path("G:/A")  # 音频主文件夹
# B_DIR = Path("G:/B")  # 视频素材文件夹
# C_DIR = Path("C:/output")  # 输出文件夹
CACHE_DIR = C_DIR / "cache"
PLAN_FILE = C_DIR / "plan.txt"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
C_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd: str) -> str:
    print(f"[cmd] {cmd}")
    result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, encoding='utf-8')
    return result.stdout.strip()

def get_duration(file_path: Path) -> float:
    cmd = f'ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 "{file_path}"'
    output = run_cmd(cmd)
    try:
        return float(output)
    except ValueError:
        print(f"[warn] 获取时长失败：{file_path.name} -> 默认设为0")
        return 0.0

def extract_audio(src: Path, dst: Path):
    cmd = f'ffmpeg -y -i "{src}" -vn -af "volume=15dB" -acodec aac -b:a 128k -ar 44100 -ac 2 "{dst}"'
    run_cmd(cmd)

def extract_video(src: Path, dst: Path, start: float, duration: float):
    cmd = f'ffmpeg -y -ss {start:.2f} -i "{src}" -t {duration:.2f} -an -c copy "{dst}"'
    run_cmd(cmd)

def merge_av(video_path: Path, audio_path: Path, output_path: Path):
    cmd = f'ffmpeg -y -i "{video_path}" -i "{audio_path}" -c copy -map 0:v:0 -map 1:a:0 "{output_path}"'
    run_cmd(cmd)

def generate_plan(a_files: List[Path], b_files: List[Path]) -> List[Tuple[str, float, List[Tuple[str, float, float]]]]:
    print("[info] 开始生成资源分配计划")
    b_video_pool = []
    for f in b_files:
        dur = get_duration(f)
        if dur > 0:
            b_video_pool.append({'file': f, 'duration': dur, 'used': 0})

    plan = []

    for a_file in a_files:
        a_duration = get_duration(a_file)
        if a_duration <= 0:
            continue

        segments = []
        remain = a_duration

        while remain > 0:
            if not b_video_pool:
                print("[warn] B 视频资源耗尽，重新随机取一个视频")
                f = random.choice(b_files)
                dur = get_duration(f)
                b_video_pool.append({'file': f, 'duration': dur, 'used': 0})

            current = b_video_pool[0]
            avail = current['duration'] - current['used']
            if avail <= 0:
                b_video_pool.pop(0)
                continue

            use_len = min(avail, remain)
            segments.append((current['file'].name, current['used'], current['used'] + use_len))
            current['used'] += use_len
            remain -= use_len

        plan.append((a_file.name, a_duration, segments))

    print("[info] 资源分配计划生成完毕")
    return plan

def write_plan(plan: List[Tuple[str, float, List[Tuple[str, float, float]]]]):
    with open(PLAN_FILE, "w", encoding="utf-8") as f:
        for name, dur, segs in plan:
            f.write(f"{name} - {dur:.2f}s\n")
            for fname, start, end in segs:
                f.write(f"    用 B 文件: {fname}, 起点: {start:.2f}s -> 终点: {end:.2f}s\n")
            f.write("\n")
    print(f"[info] 已写入资源分配计划到 {PLAN_FILE}")

def async_copy_file(src: Path, dst: Path):
    def copy():
        print(f"[copy] 开始复制 {src.name} -> {dst}")
        dst.write_bytes(src.read_bytes())
        print(f"[copy] 复制完成 {dst.name}")
    threading.Thread(target=copy).start()

def process_plan(plan):
    for idx, (a_name, dur, segs) in enumerate(plan):
        a_path = A_DIR / a_name
        audio_path = CACHE_DIR / f"{a_path.stem}_audio.aac"
        extract_audio(a_path, audio_path)

        temp_video_path = CACHE_DIR / f"{a_path.stem}_video.mp4"
        temp_list = []

        for i, (b_name, start, end) in enumerate(segs):
            duration = end - start
            part_path = CACHE_DIR / f"{a_path.stem}_part_{i}.mp4"
            extract_video(B_DIR / b_name, part_path, start, duration)
            temp_list.append(part_path)

        concat_list = CACHE_DIR / f"{a_path.stem}_concat.txt"
        with open(concat_list, "w", encoding="utf-8") as f:
            for p in temp_list:
                f.write(f"file '{p.as_posix()}'\n")

        merged_video = CACHE_DIR / f"{a_path.stem}_merged.mp4"
        run_cmd(f'ffmpeg -y -f concat -safe 0 -i "{concat_list}" -c copy "{merged_video}"')

        final_output = C_DIR / f"{a_path.stem}.mp4"
        merge_av(merged_video, audio_path, final_output)

        print(f"[done] 已完成：{final_output.name}")

        # 删除缓存
        for p in temp_list + [audio_path, merged_video, concat_list]:
            if p.exists():
                p.unlink()
        print(f"[clean] 缓存清理完成：{a_path.stem}\n")

# 运行主流程
if __name__ == "__main__":
    print("[start] 音视频合成任务开始")

    a_files = sorted([f for f in A_DIR.glob("*.mp4")])
    b_files = sorted([f for f in B_DIR.glob("*.mp4")])
    plan = generate_plan(a_files, b_files)
    write_plan(plan)
    process_plan(plan)

    print("[end] 所有任务已完成")
